# Create and run a local RAG pipeline from scratch

## What is RAG?

**RAG** stands for Retrieval-Augmented Generation.

The goal of RAG is to take information and pass it to an LLM so it can generate outputs based on that info.

* **Retrieval** - Find relevant information given a query, e.g. "what are macronutrients?" -> retrieve passages of text related to macronutrients.
* **Augmented** - We want to take the relevant info from retrieval and augment our input (prompt) to an LLM with that information.
* **Generation** -  Take the first two steps and pass them to an LLM for generative outputs.

## Why RAG?

The main goal of RAG is to improve the generation outputs of LLMs.

1. Prevent hallucinations - LLMs are good at generation good looking text, but that does not mean that it is factually correct.
    * RAG can help LLMs generate information based on relevant passages that are factual
2. Work with custom data - Many base LLMs are trained with internet-scale data (means good understanding with language in general), but means the responses can be generic in nature.
    * RAG helps create specific responses based on specific documents

## Overview of the RAG pipeline

1. Open a PDF document.
2. Format the text of the PDF textbook ready for an embedding model.
3. Embed all the chunks of text in the textbook and turn them into numerical representations which can be stored for use later.
4. Build a retrieval that uses vector search to find relevant chunks of text given a query.
5. Create a prompt that incorporates the retrieved pieces of text.
6. Generate an answer to the query based on the passages of the textbook that were retrieved with an LLM.

**Steps 1-3:** Document preprocessing and embedding creation

**Steps 4-6:** Retrieval and generation (search and answer)

## 1. Document/text processing and embedding creation

Items needed:
* PDF document of choice
* Embedding moodel of choice

Steps:
1. Import the PDF document
2. Process text for embedding (e.g. split into chunks of sentences)
3. Embed text chunks with embedding model
4. Save embeddings to a file for later retrieval

### Import PDF document

In [2]:
import os # file path handling
import requests # download things off the internet

# Get PDF document path
pdf_path = "human-nutrition-text.pdf"

# Download pdf
if not os.path.exists(pdf_path):
    print(f"[INFO] File doesn't exist, downloading...")

    # URL of pdf
    url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"

    # local filename
    filename = pdf_path

    # Send a GET request to the URL
    response = requests.get(url)

    # Check if the request was successful
    if response.status_code == 200:
        # Open the file and save it
        with open(filename, "wb") as file:
            file.write(response.content)
        
        print(f"[INFO] The file has been downloaded and saved as {filename}")
    else:
        print(f"[ERR] Failed to download file. Status code: {response.status_code}")
else:
    print(f"[INFO] File {pdf_path} already exists.")

[INFO] File human-nutrition-text.pdf already exists.


### Open PDF

In [3]:
import fitz # PyMuPDF -> used to read pdf files
from tqdm.auto import tqdm # progress bars

def text_formatter(text: str) -> str:
    """Performs minor formatting on text"""
    
    # remove newlines and extra whitespace chars
    cleaned_text = text.replace("\n", " ").strip()

    return cleaned_text

def read_pdf(path: str) -> list[dict]:
    doc = fitz.open(path)

    pages = []

    # Loop through all the pages in the pdf
    for page_num, page in tqdm(enumerate(doc)):
        text = page.get_text() # get text of current page
        text = text_formatter(text) # format text

        # append all important info about the current page
        pages.append({
            "page_number" : page_num - 41,
            "char_count": len(text),
            "word_count": len(text.split(" ")),
            "sentence_count": len(text.split(". ")),
            "token_count": len(text) / 4, # 1 token ~= 4 chars
            "text": text
        })

    return pages

#### Key term

| Term | Description |
| ----- | ----- | 
| **Token** | A sub-word piece of text. For example, "hello, world!" could be split into ["hello", ",", "world", "!"]. A token can be a whole word,<br> part of a word or group of punctuation characters. 1 token ~= 4 characters in English, 100 tokens ~= 75 words.<br> Text gets broken into tokens before being passed to an LLM. |

In [4]:
pages = read_pdf(path=pdf_path)

import random

random.sample(pages, k=3)

0it [00:00, ?it/s]

[{'page_number': 642,
  'char_count': 1248,
  'word_count': 235,
  'sentence_count': 11,
  'token_count': 312.0,
  'text': 'Age Group  RDA (mg/ day)  UL from non-food sources (mg/ day)  Infants (0–6 months)  30*  –  Infants (6–12 months)  75*  –  Children (1–3 years)  80  65  Children (4–8 years)  130  110  Children (9–13 years)  240  350  Adolescents (14–18  years)  410  350  Adults (19–30 years)  400  350  Adults (> 30 years)  420  350  * denotes Adequate  Intake  Source:  Dietary Supplement Fact Sheet: Magnesium. National  Institutes  of  Health,  Office  of  Dietary  Supplements.  http://ods.od.nih.gov/factsheets/Magnesium- HealthProfessional/. Updated July 13, 2009. Accessed November 12,  2017.  Dietary Sources of Magnesium  Magnesium is part of the green pigment, chlorophyll, which is vital  for photosynthesis in plants; therefore green leafy vegetables are  a good dietary source for magnesium. Magnesium is also found in  high concentrations in fish, dairy products, meats, whole 

In [5]:
import pandas as pd

df = pd.DataFrame(pages) # convert the dict into a dataframe
df.head()

,page_number,char_count,word_count,sentence_count,token_count,text
0,-41,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,1,0.00,
2,-39,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,147,3,199.25,Contents Preface University of Hawai‘i at Mā...


In [6]:
df.describe().round(2) # generate a statistical summary of the data

,page_number,char_count,word_count,sentence_count,token_count
count,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,199.50,10.52,287.00
std,348.86,560.38,95.83,6.55,140.10
min,-41.00,0.00,1.00,1.00,0.00
25%,260.75,762.00,134.00,5.00,190.50
50%,562.50,1231.50,216.00,10.00,307.88
75%,864.25,1603.50,272.00,15.00,400.88
max,1166.00,2308.00,430.00,39.00,577.00


#### Why does the token count matter?

Token count is important to think about because:
1. Embedding models don't deal with infinite tokens.
2. LLMs don't deal with infinite tokens.

The embedding model we will use (`all-mpnet-base-v2`) is trained to embed sequences of 384 tokens.

As for LLMs, they can't accept infinite tokens in their **context window**.

#### Key term

| Term | Description |
| ----- | ----- | 
| **LLM context window** | The number of tokens a LLM can accept as input. For example, as of March 2024, GPT-4 has a default context window of 32k tokens<br> (about 96 pages of text) but can go up to 128k if needed. A recent open-source LLM from Google, Gemma (March 2024) has a context<br> window of 8,192 tokens (about 24 pages of text). A higher context window means an LLM can accept more relevant information<br> to assist with a query. For example, in a RAG pipeline, if a model has a larger context window, it can accept more reference items<br> from the retrieval system to aid with its generation. |

### Preprocess text for chunking

Split pages into sentences by either:
1. Splitting on `". "`
2. Use NLP library (e.g. `spaCy` and `nltk`)

In [ ]:
from spacy.lang.en import English

nlp = English() # create a blank English NLP model

# Add a sentencizer pipeline -> turns text into sentences
nlp.add_pipe("sentencizer")

# Create a document instance as an example
doc = nlp("This is the first sentence. This is the second sentence. I like cats.")
assert len(list(doc.sents)) == 3

# Print out the sentences split
list(doc.sents)

[This is the first sentence?, This is the second sentence., I like cats.]

In [11]:
pages[67]

{'page_number': 26,
 'char_count': 1799,
 'word_count': 321,
 'sentence_count': 21,
 'token_count': 449.75,
 'text': 'Factors that Drive Food Choices  Along with these influences, a number of other factors affect the  dietary choices individuals make, including:  • Taste, texture, and appearance. Individuals have a wide range  of tastes which influence their food choices, leading some to  dislike milk and others to hate raw vegetables. Some foods that  are very healthy, such as tofu, may be unappealing at first to  many people. However, creative cooks can adapt healthy foods  to meet most people’s taste.  • Economics. Access to fresh fruits and vegetables may be scant,  particularly for those who live in economically disadvantaged  or remote areas, where cheaper food options are limited to  convenience stores and fast food.  • Early food experiences. People who were not exposed to  different foods as children, or who were forced to swallow  every last bite of overcooked vegetables, may

In [12]:
for item in tqdm(pages):
    # Apply the pipeline to the text and get the sentences
    item["sentences"] = list(nlp(item["text"]).sents)

    # Convert all sentences to strings (default is a spaCy datatype)
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]

    # Count the sentences
    item["sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [17]:
random.sample(pages, k=1)

[{'page_number': 761,
  'char_count': 1398,
  'word_count': 291,
  'sentence_count': 13,
  'token_count': 349.5,
  'text': 'Table 12.6 The Different Categories from the Pacific  Energy  Nutrient-dense  foods  Protective Foods  Body-building  Foods  Types of  Foods  Foods that are  both high in  calories and high  in nutrients  Fruits and vegetables  Protein-rich  foods  Description  The  recommendation  is that these foods  should be  included in all  meals  contributing to  about half of the  food you  consume each  day.  The foods in this  group are high in  vitamins and minerals.  These foods are  recommended to be   included in all meals  and snacks  contributing about  one third of the food  consumed each day.  The foods in  this group are  high in  protein and is  recommended  to be eaten  twice a day in  small  amounts.  The recommendation is that these foods should be included in all  meals contributing to about half of the food you consume each day.  The foods in this group ar

In [18]:
df = pd.DataFrame(pages)
df.describe().round(2)

,page_number,char_count,word_count,sentence_count,token_count,sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,199.50,10.52,287.00,10.32
std,348.86,560.38,95.83,6.55,140.10,6.30
min,-41.00,0.00,1.00,1.00,0.00,0.00
25%,260.75,762.00,134.00,5.00,190.50,5.00
50%,562.50,1231.50,216.00,10.00,307.88,10.00
75%,864.25,1603.50,272.00,15.00,400.88,15.00
max,1166.00,2308.00,430.00,39.00,577.00,28.00
